# 🧠 Medical Image ROI Extraction — Model Training
## EfficientNet-B3 + GradCAM++ | X-Ray · CT · MRI Brain Tumor

**All memory, stability, and accuracy fixes applied:**

| # | Fix | Why |
|---|-----|-----|
| 1 | `memmap` dataset — zero full-load RAM | Avoids OOM during training |
| 2 | CPU-safe AMP (`USE_AMP = cuda only`) | No AMP crash on CPU |
| 3 | GradScaler only on CUDA | Avoids CPU GradScaler error |
| 4 | `autocast` context guard | Works on both CPU and GPU |
| 5 | val_loss early stopping (patience=7) | Prevents overfit |
| 6 | MixUp + label smoothing 0.15 | No overconfident 100% predictions |
| 7 | Dropout 0.5/0.3 | Stronger regularisation |
| 8 | LR warmup 3 epochs | Stable early training |
| 9 | Adaptive batch size (CPU=16, GPU=32) | Prevents CPU RAM spike |
| 10 | `num_workers=0` always | Avoids Windows multiprocess crash |

In [1]:
# ── CELL 1 — Imports & Config ────────────────────────────────────────────────
import os, gc, sys, time, json, warnings
import numpy as np
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — works on CPU-only servers too
import matplotlib.pyplot as plt
import matplotlib.cm as mpl_cm
import cv2
from pathlib import Path
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from torchvision.models import efficientnet_b3, EfficientNet_B3_Weights
from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, roc_curve, ConfusionMatrixDisplay)
from sklearn.preprocessing import label_binarize
from sklearn.metrics import precision_recall_fscore_support
from PIL import Image as PILImage
warnings.filterwarnings('ignore')

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# ── Device & AMP ──────────────────────────────────────────────────────────────
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP    = (DEVICE.type == 'cuda')   # AMP only on CUDA — CPU has no benefit
PIN_MEMORY = (DEVICE.type == 'cuda')   # pin_memory requires CUDA

# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_DIR  = Path('preprocessed')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

# ── Hyperparameters ───────────────────────────────────────────────────────────
IMG_SIZE   = 224
# Adaptive batch size: smaller on CPU to avoid RAM spike
BATCH_SIZE = 32 if DEVICE.type == 'cuda' else 16
EPOCHS     = 40
LR         = 3e-4
PATIENCE   = 7    # early stop patience on val_loss
WARMUP_EP  = 3    # linear LR warmup epochs

# Per-modality unfreeze blocks
UNFREEZE_BLOCKS = {
    'xray': ('features.6', 'features.7', 'features.8'),
    'ct':   ('features.7', 'features.8'),
    'mri':  ('features.7', 'features.8'),
}

# ── Summary ───────────────────────────────────────────────────────────────────
print(f'Device     : {DEVICE}')
print(f'AMP        : {USE_AMP}')
print(f'Batch size : {BATCH_SIZE}')
if DEVICE.type == 'cuda':
    print(f'GPU        : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f'VRAM       : {vram:.1f} GB')

# ── Verify preprocessed splits exist ─────────────────────────────────────────
print('\nVerifying preprocessed splits...')
all_ok = True
for mod in ['xray', 'ct', 'mri']:
    for split in ['train', 'val', 'test']:
        for fname in ['X.npy', 'y.npy']:
            fp = DATA_DIR / mod / split / fname
            if not fp.exists():
                print(f'  ❌ MISSING: {fp}')
                all_ok = False
if all_ok:
    print('  ✅ All splits found. Ready to train.')
else:
    print('  ⚠️  Run preprocessing.ipynb first.')

Device     : cuda
AMP        : True
Batch size : 32
GPU        : NVIDIA GeForce RTX 3050 Laptop GPU
VRAM       : 4.0 GB

Verifying preprocessed splits...
  ✅ All splits found. Ready to train.


In [2]:
# ── CELL 2 — Dataset & DataLoader ────────────────────────────────────────────
# Uses np.memmap so only individual samples are loaded into RAM per __getitem__.
# This is what allows training on large datasets without OOM.

IMAGENET_NORM = T.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])

def get_train_transform():
    return T.Compose([
        T.RandomHorizontalFlip(p=0.5),
        T.RandomRotation(degrees=12),
        T.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1),
        T.RandomAffine(degrees=0, translate=(0.05, 0.05)),
        T.ToTensor(),
        IMAGENET_NORM,
    ])

def get_eval_transform():
    return T.Compose([T.ToTensor(), IMAGENET_NORM])


class MedicalImageDataset(Dataset):
    """
    Memory-mapped dataset: X.npy is opened as a memmap.
    Only the individual sample requested by __getitem__ is paged into RAM.
    Safe for datasets of any size.
    """
    def __init__(self, modality: str, split: str, transform=None):
        base   = DATA_DIR / modality / split
        x_path = str(base / 'X.npy')
        y_path = str(base / 'y.npy')

        self.y = np.load(y_path).astype(np.int64)   # labels fit in RAM easily
        n      = len(self.y)

        # Validate memmap size before opening
        raw = np.memmap(x_path, dtype='float32', mode='r')
        assert raw.size == n * IMG_SIZE * IMG_SIZE * 3, \
            (f'Shape mismatch in {x_path}: '
             f'has {raw.size} elements, expected {n * IMG_SIZE * IMG_SIZE * 3}. '
             f'Re-run preprocessing.')
        del raw

        # Open as memmap (no RAM allocated for data yet)
        self.X         = np.memmap(x_path, dtype='float32', mode='r',
                                   shape=(n, IMG_SIZE, IMG_SIZE, 3))
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # copy=True forces a real copy from memmap → only this sample in RAM
        img   = np.array(self.X[idx], copy=True)   # HWC float32 [0,1]
        label = int(self.y[idx])
        if self.transform:
            img = (img * 255).astype(np.uint8)
            img = self.transform(PILImage.fromarray(img))
        else:
            img = torch.from_numpy(
                np.ascontiguousarray(img)).permute(2, 0, 1).float()
        return img, label


def make_loaders(modality: str):
    train_ds = MedicalImageDataset(modality, 'train', get_train_transform())
    val_ds   = MedicalImageDataset(modality, 'val',   get_eval_transform())
    test_ds  = MedicalImageDataset(modality, 'test',  get_eval_transform())

    kw = dict(
        batch_size       = BATCH_SIZE,
        num_workers      = 0,      # 0 = no subprocess — safe on Windows & CPU
        pin_memory       = PIN_MEMORY,
        persistent_workers = False,
    )
    classes = np.load(DATA_DIR / modality / 'classes.npy', allow_pickle=True)

    print(f'  [{modality}] Train={len(train_ds)}, Val={len(val_ds)}, Test={len(test_ds)}')
    print(f'           Classes ({len(classes)}): {list(classes)}')

    return (DataLoader(train_ds, shuffle=True,  **kw),
            DataLoader(val_ds,   shuffle=False, **kw),
            DataLoader(test_ds,  shuffle=False, **kw),
            classes)


# ── MixUp ─────────────────────────────────────────────────────────────────────
def mixup_batch(X, y, alpha=0.3):
    """MixUp: blend pairs. lam >= 0.5 always so dominant class is preserved."""
    if alpha <= 0:
        return X, y, y, 1.0
    lam = float(np.random.beta(alpha, alpha))
    lam = max(lam, 1.0 - lam)
    idx = torch.randperm(X.size(0), device=X.device)
    return lam * X + (1.0 - lam) * X[idx], y, y[idx], lam


def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1.0 - lam) * criterion(pred, y_b)


print('✅ Dataset & loaders ready.')
print(f'   Batch size: {BATCH_SIZE}')

✅ Dataset & loaders ready.
   Batch size: 32


In [3]:
# ── CELL 3 — ROIClassifier (EfficientNet-B3 + GradCAM++) ─────────────────────

class ROIClassifier(nn.Module):
    """
    EfficientNet-B3 fine-tuned for medical image classification.
    Selective layer unfreezing + two-stage dropout head.
    GradCAM++ hooks registered on last feature block.
    """
    def __init__(self, num_classes: int, dropout: float = 0.5,
                 unfreeze_blocks=None):
        super().__init__()
        if unfreeze_blocks is None:
            unfreeze_blocks = ('features.6', 'features.7', 'features.8')

        base = efficientnet_b3(weights=EfficientNet_B3_Weights.DEFAULT)

        # Freeze all, then unfreeze selected blocks
        for p in base.parameters():
            p.requires_grad = False
        for name, p in base.named_parameters():
            if any(blk in name for blk in unfreeze_blocks):
                p.requires_grad = True

        self.features  = base.features
        self.avgpool   = base.avgpool
        in_feat        = base.classifier[1].in_features  # 1536 for B3

        # Two-layer bottleneck head with aggressive dropout
        self.classifier = nn.Sequential(
            nn.Dropout(p=dropout),          # 0.5
            nn.Linear(in_feat, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=dropout * 0.6),    # 0.3
            nn.Linear(256, num_classes),
        )

        # GradCAM++ hooks
        self._grads = None
        self._acts  = None
        self.features[-1].register_forward_hook(self._hook_act)
        self.features[-1].register_full_backward_hook(self._hook_grad)

        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        total     = sum(p.numel() for p in self.parameters())
        print(f'  Trainable: {trainable:,}/{total:,} '
              f'({100*trainable/total:.1f}%)  Blocks: {unfreeze_blocks}')

    def _hook_act(self, m, inp, out):
        self._acts  = out.detach()

    def _hook_grad(self, m, g_in, g_out):
        self._grads = g_out[0].detach()

    def forward(self, x):
        feat   = self.features(x)
        pooled = self.avgpool(feat).flatten(1)
        return self.classifier(pooled)

    def gradcam_plus_plus(self, img_tensor: torch.Tensor,
                           class_idx: int = None):
        """
        GradCAM++ inference. Works on both CPU and CUDA.
        - No @torch.no_grad() — backward() needs gradients.
        - Gaussian blur + 30% noise threshold for clean ROI maps.
        Returns: cam [H,W] float32 [0,1], class_idx int, probs [C] float32
        """
        self.eval()
        device = next(self.parameters()).device
        inp    = img_tensor.unsqueeze(0).to(device)

        with torch.enable_grad():
            inp    = inp.requires_grad_(True)
            output = self(inp)
            if class_idx is None:
                class_idx = int(output.argmax(dim=1).item())
            self.zero_grad()
            output[0, class_idx].backward()

        grads = self._grads[0]   # (C, H, W)
        acts  = self._acts[0]    # (C, H, W)

        # GradCAM++ weights
        gsq   = grads ** 2
        gcb   = grads ** 3
        denom = 2.0 * gsq + (acts * gcb).sum(dim=(1, 2), keepdim=True)
        denom = torch.where(denom != 0, denom, torch.ones_like(denom))
        alpha   = gsq / denom
        weights = (alpha * torch.relu(grads)).sum(dim=(1, 2))
        cam     = (weights[:, None, None] * acts).sum(0)
        cam     = torch.relu(cam).cpu().numpy()

        # Normalise → resize → smooth → threshold
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = cv2.resize(cam.astype(np.float32), (IMG_SIZE, IMG_SIZE))
        cam = cv2.GaussianBlur(cam, (11, 11), 0)
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        cam = np.where(cam < 0.30, 0.0, cam).astype(np.float32)
        if cam.max() > 0:
            cam = cam / cam.max()

        probs = torch.softmax(output, dim=1)[0].detach().cpu().numpy()

        if device.type == 'cuda':
            torch.cuda.empty_cache()
        return cam, class_idx, probs


print('✅ ROIClassifier defined.')

✅ ROIClassifier defined.


In [4]:
# ── CELL 4 — Training Functions ──────────────────────────────────────────────
# CPU/GPU safe: autocast and GradScaler are guarded by USE_AMP.

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = correct = total = 0

    for X, y in loader:
        X = X.float().to(DEVICE, non_blocking=PIN_MEMORY)
        y = y.to(DEVICE, non_blocking=PIN_MEMORY)

        # MixUp
        X, y_a, y_b, lam = mixup_batch(X, y, alpha=0.3)

        optimizer.zero_grad(set_to_none=True)

        # autocast only on CUDA
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                out  = model(X)
                loss = mixup_criterion(criterion, out, y_a, y_b, lam)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            out  = model(X)
            loss = mixup_criterion(criterion, out, y_a, y_b, lam)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y_a).sum().item()
        total      += X.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss = correct = total = 0
    all_probs, all_labels = [], []

    for X, y in loader:
        X = X.float().to(DEVICE, non_blocking=PIN_MEMORY)
        y = y.to(DEVICE, non_blocking=PIN_MEMORY)

        if USE_AMP:
            with torch.amp.autocast('cuda'):
                out  = model(X)
                loss = criterion(out, y)
        else:
            out  = model(X)
            loss = criterion(out, y)

        total_loss += loss.item() * X.size(0)
        correct    += (out.argmax(1) == y).sum().item()
        total      += X.size(0)
        all_probs.append(torch.softmax(out, 1).cpu().numpy())
        all_labels.append(y.cpu().numpy())

    return (total_loss / total,
            correct / total,
            np.concatenate(all_probs),
            np.concatenate(all_labels))


def warmup_lr(optimizer, epoch, warmup_epochs, base_lr):
    """Linear LR warmup to prevent instability in first few epochs."""
    if epoch <= warmup_epochs:
        for pg in optimizer.param_groups:
            pg['lr'] = base_lr * (epoch / warmup_epochs)


print('✅ Training functions ready.')

✅ Training functions ready.


In [5]:
# ── CELL 5 — train_modality() ────────────────────────────────────────────────

def train_modality(modality: str, epochs: int = EPOCHS, lr: float = LR):
    print(f'\n' + '='*65)
    print(f'  Training: {modality.upper()}')
    print('='*65)

    train_loader, val_loader, test_loader, classes = make_loaders(modality)
    num_classes = len(classes)

    if num_classes < 2:
        print(f'  ⚠️  SKIPPING — only {num_classes} class. ROI-only mode.')
        return None, None, classes, None, None, test_loader

    model = ROIClassifier(
        num_classes,
        dropout        = 0.5,
        unfreeze_blocks = UNFREEZE_BLOCKS[modality]
    ).to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.15)
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr, weight_decay=2e-4
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(epochs - WARMUP_EP, 1), eta_min=1e-6
    )
    # GradScaler only on CUDA — on CPU it raises an error
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

    history = {'train_loss': [], 'val_loss': [],
               'train_acc':  [], 'val_acc':  []}
    best_val_loss  = float('inf')
    best_val_acc   = 0.0
    patience_count = 0
    best_path      = MODEL_DIR / f'{modality}_best.pth'

    print(f'  Criterion : CrossEntropyLoss(label_smoothing=0.15)')
    print(f'  Dropout   : 0.5 / 0.3')
    print(f'  Warmup    : {WARMUP_EP} epochs  |  Patience: {PATIENCE} (val_loss)')
    print()

    for epoch in range(1, epochs + 1):
        t0 = time.time()

        # LR warmup (overrides scheduler for first WARMUP_EP epochs)
        if epoch <= WARMUP_EP:
            warmup_lr(optimizer, epoch, WARMUP_EP, lr)

        tr_loss, tr_acc       = train_one_epoch(
            model, train_loader, criterion, optimizer, scaler)
        vl_loss, vl_acc, _, _ = evaluate(model, val_loader, criterion)

        if epoch > WARMUP_EP:
            scheduler.step()

        # Cleanup each epoch
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)

        improved = vl_loss < best_val_loss
        if improved:
            best_val_loss  = vl_loss
            best_val_acc   = vl_acc
            patience_count = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'classes':          list(classes),
                'num_classes':      num_classes,
                'modality':         modality,
                'img_size':         IMG_SIZE,
                'best_val_loss':    best_val_loss,
                'best_val_acc':     best_val_acc,
                'imagenet_mean':    [0.485, 0.456, 0.406],
                'imagenet_std':     [0.229, 0.224, 0.225],
            }, best_path)
        else:
            patience_count += 1

        gap    = (tr_acc - vl_acc) * 100
        lr_now = optimizer.param_groups[0]['lr']
        marker = ' ✓' if improved else f' (patience {patience_count}/{PATIENCE})'
        elapsed = time.time() - t0
        print(f'  Ep {epoch:3d}/{epochs} | '
              f'Loss {tr_loss:.4f}/{vl_loss:.4f} | '
              f'Acc {tr_acc*100:.1f}%/{vl_acc*100:.1f}% '
              f'[gap {gap:+.1f}%] | '
              f'LR {lr_now:.2e} | '
              f'{elapsed:.0f}s{marker}')

        if patience_count >= PATIENCE:
            print(f'  ⏹  Early stop at epoch {epoch} — val_loss stagnant.')
            break

    # ── Load best checkpoint & evaluate on test set ───────────────────────────
    ckpt = torch.load(best_path, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    _, test_acc, test_probs, test_labels = evaluate(model, test_loader, criterion)

    print(f'\n  ✅ Best val_loss  : {best_val_loss:.4f}')
    print(f'     Best val_acc   : {best_val_acc*100:.2f}%')
    print(f'     Test accuracy  : {test_acc*100:.2f}%')

    final_gap = (history['train_acc'][-1] - history['val_acc'][-1]) * 100
    best_ep   = int(np.argmin(history['val_loss'])) + 1
    health = ('✅ well-generalised' if final_gap <= 5
              else '🟡 mild overfit' if final_gap <= 10
              else '⚠️  overfitting')
    print(f'     Final train/val gap: {final_gap:+.1f}%  {health}')
    print(f'     Best epoch: {best_ep}/{len(history["val_loss"])}')

    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()
    return model, history, classes, test_probs, test_labels, test_loader


print('✅ train_modality() ready.')

✅ train_modality() ready.


In [6]:
# ── CELL 6 — Train All Modalities ────────────────────────────────────────────
results = {}

for modality in ['xray', 'ct', 'mri']:
    model, history, classes, test_probs, test_labels, test_loader = \
        train_modality(modality)
    results[modality] = {
        'model':       model,
        'history':     history,
        'classes':     classes,
        'test_probs':  test_probs,
        'test_labels': test_labels,
        'test_loader': test_loader,
        'trained':     model is not None,
    }
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    gc.collect()

print('\n🎉 Training complete.')
print()
for mod, r in results.items():
    status = '✅ Trained' if r['trained'] else '⚠️  Skipped'
    print(f'  {mod.upper():6s}: {status}')


  Training: XRAY
  [xray] Train=14821, Val=3172, Test=3172
           Classes (4): ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
  Trainable: 8,900,578/11,090,732 (80.3%)  Blocks: ('features.6', 'features.7', 'features.8')
  Criterion : CrossEntropyLoss(label_smoothing=0.15)
  Dropout   : 0.5 / 0.3
  Warmup    : 3 epochs  |  Patience: 7 (val_loss)

  Ep   1/40 | Loss 0.9603/0.6977 | Acc 74.0%/87.7% [gap -13.7%] | LR 1.00e-04 | 224s ✓
  Ep   2/40 | Loss 0.8489/0.6700 | Acc 83.3%/91.0% [gap -7.7%] | LR 2.00e-04 | 184s ✓
  Ep   3/40 | Loss 0.8145/0.6371 | Acc 85.4%/92.0% [gap -6.6%] | LR 3.00e-04 | 172s ✓
  Ep   4/40 | Loss 0.7962/0.6096 | Acc 86.5%/93.3% [gap -6.8%] | LR 2.99e-04 | 176s ✓
  Ep   5/40 | Loss 0.7888/0.6198 | Acc 87.4%/93.6% [gap -6.1%] | LR 2.98e-04 | 169s (patience 1/7)
  Ep   6/40 | Loss 0.7640/0.6147 | Acc 88.9%/93.3% [gap -4.4%] | LR 2.95e-04 | 171s (patience 2/7)
  Ep   7/40 | Loss 0.7561/0.6119 | Acc 89.6%/92.9% [gap -3.3%] | LR 2.91e-04 | 172s (patience 3/

In [7]:
# ── CELL 7 — Training Curves ─────────────────────────────────────────────────
trained_mods = [m for m in ['xray', 'ct', 'mri'] if results[m]['trained']]
n = len(trained_mods)

if n == 0:
    print('No modalities trained — nothing to plot.')
else:
    ncols = n
    fig, axes = plt.subplots(2, ncols, figsize=(7 * ncols, 10))
    if ncols == 1:
        axes = axes[:, np.newaxis]   # ensure 2D indexing
    fig.suptitle('Training Curves (val_loss early stopping | MixUp | label smoothing 0.15)',
                 fontsize=13, fontweight='bold')

    for col, modality in enumerate(trained_mods):
        h    = results[modality]['history']
        eps  = range(1, len(h['train_loss']) + 1)
        best = int(np.argmin(h['val_loss'])) + 1

        ax0 = axes[0, col]
        ax0.plot(eps, h['train_loss'], label='Train', color='royalblue')
        ax0.plot(eps, h['val_loss'],   label='Val',   color='tomato')
        ax0.axvline(best, color='green', ls='--', lw=1.5, label=f'Best ({best})')
        ax0.set_title(f'{modality.upper()} — Loss', fontweight='bold')
        ax0.legend()
        ax0.set_xlabel('Epoch')

        ax1 = axes[1, col]
        tr_acc = [a * 100 for a in h['train_acc']]
        vl_acc = [a * 100 for a in h['val_acc']]
        ax1.plot(eps, tr_acc, label='Train', color='royalblue')
        ax1.plot(eps, vl_acc, label='Val',   color='tomato')
        ax1.fill_between(eps, vl_acc, tr_acc,
                         alpha=0.15, color='red', label='Gap')
        ax1.axvline(best, color='green', ls='--', lw=1.5)
        ax1.set_title(f'{modality.upper()} — Accuracy (%)', fontweight='bold')
        ax1.legend()
        ax1.set_xlabel('Epoch')
        ax1.set_ylim([0, 100])

    plt.tight_layout()
    out_path = 'models/training_curves.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved → {out_path}')

✅ Saved → models/training_curves.png


In [8]:
# ── CELL 8 — Evaluation: Classification Report, Confusion Matrix, ROC ────────

def plot_confusion_matrix(labels, preds, classes, modality):
    disp = ConfusionMatrixDisplay(
        confusion_matrix(labels, preds), display_labels=classes)
    fig, ax = plt.subplots(figsize=(6, 5))
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{modality.upper()} — Confusion Matrix', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'models/{modality}_confusion_matrix.png', dpi=120)
    plt.show()


def plot_roc_curves(labels, probs, classes, modality):
    n_cls  = len(classes)
    y_bin  = label_binarize(labels, classes=range(n_cls))
    colors = plt.cm.Set1(np.linspace(0, 0.8, n_cls))
    auc_scores = []
    fig, ax = plt.subplots(figsize=(7, 5))

    for i, (cls, color) in enumerate(zip(classes, colors)):
        try:
            fpr, tpr, _ = roc_curve(y_bin[:, i], probs[:, i])
            auc         = roc_auc_score(y_bin[:, i], probs[:, i])
            auc_scores.append(auc)
            ax.plot(fpr, tpr, color=color, lw=2,
                    label=f'{cls} (AUC={auc:.3f})')
        except Exception as e:
            print(f'  ROC skip {cls}: {e}')

    ax.plot([0, 1], [0, 1], 'k--', lw=1)
    ax.set_xlabel('FPR')
    ax.set_ylabel('TPR')
    ax.set_title(f'{modality.upper()} — ROC Curves', fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    plt.savefig(f'models/{modality}_roc_curves.png', dpi=120)
    plt.show()
    return auc_scores


for modality in ['xray', 'ct', 'mri']:
    r = results[modality]
    if not r['trained']:
        print(f'\n{modality.upper()} — Skipped.')
        continue

    preds = np.argmax(r['test_probs'], axis=1)
    print(f'\n' + '='*55)
    print(f'  {modality.upper()} — Classification Report')
    print('='*55)
    print(classification_report(
        r['test_labels'], preds, target_names=r['classes']))

    plot_confusion_matrix(r['test_labels'], preds, r['classes'], modality)
    auc = plot_roc_curves(
        r['test_labels'], r['test_probs'], r['classes'], modality)
    if auc:
        print(f'  Mean AUC: {np.mean(auc):.4f}')


  XRAY — Classification Report
                 precision    recall  f1-score   support

          COVID       0.98      0.97      0.97       542
   Lung_Opacity       0.95      0.89      0.92       901
         Normal       0.93      0.97      0.95      1528
Viral Pneumonia       0.98      0.96      0.97       201

       accuracy                           0.95      3172
      macro avg       0.96      0.95      0.95      3172
   weighted avg       0.95      0.95      0.95      3172

  Mean AUC: 0.9922

  CT — Classification Report
                         precision    recall  f1-score   support

         adenocarcinoma       0.89      0.47      0.62       120
   large cell carcinoma       0.47      0.88      0.62        51
                 normal       0.93      0.98      0.95        54
squamous cell carcinoma       0.76      0.83      0.79        90

               accuracy                           0.73       315
              macro avg       0.76      0.79      0.75       315
   

In [9]:
# ── CELL 9 — GradCAM++ ROI Visualisation ─────────────────────────────────────
IMAGENET_MEAN_VIS = np.array([0.485, 0.456, 0.406])
IMAGENET_STD_VIS  = np.array([0.229, 0.224, 0.225])


def denormalize(tensor_img: torch.Tensor) -> np.ndarray:
    img = tensor_img.cpu().permute(1, 2, 0).numpy()
    return np.clip(img * IMAGENET_STD_VIS + IMAGENET_MEAN_VIS, 0, 1)


def overlay_cam(orig_float: np.ndarray, cam: np.ndarray,
                alpha: float = 0.5) -> np.ndarray:
    heat = cv2.applyColorMap((cam * 255).astype(np.uint8),
                              cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    mask = cam[:, :, np.newaxis]
    return np.clip(alpha * mask * heat
                   + (1.0 - alpha * mask) * orig_float, 0, 1)


def visualize_roi(modality: str, n_samples: int = 4):
    r       = results[modality]
    classes = r['classes']

    if r['trained']:
        model = r['model']
    else:
        ckpt_path = MODEL_DIR / f'{modality}_best.pth'
        if ckpt_path.exists():
            ckpt  = torch.load(ckpt_path, map_location=DEVICE,
                               weights_only=False)
            model = ROIClassifier(
                ckpt['num_classes'],
                unfreeze_blocks=UNFREEZE_BLOCKS[modality]
            ).to(DEVICE)
            model.load_state_dict(ckpt['model_state_dict'])
        else:
            print(f'  ⚠️  {modality}: no checkpoint, using random weights (demo)')
            model = ROIClassifier(
                max(len(classes), 2),
                unfreeze_blocks=UNFREEZE_BLOCKS[modality]
            ).to(DEVICE)

    print(f'\n🔍 {modality.upper()} — GradCAM++ ROI')

    X_batch, y_batch = next(iter(r['test_loader']))
    n_samples = min(n_samples, len(X_batch))
    X_batch   = X_batch[:n_samples].float()

    fig, axes = plt.subplots(n_samples, 3,
                              figsize=(12, 4 * n_samples))
    if n_samples == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(
        f'{modality.upper()} — GradCAM++  |  Original  |  Heatmap  |  Overlay',
        fontsize=13, fontweight='bold')

    for i in range(n_samples):
        img_t    = X_batch[i]
        true_lbl = int(y_batch[i].item())
        cam, pred_cls, probs = model.gradcam_plus_plus(img_t)
        orig = denormalize(img_t)
        over = overlay_cam(orig, cam)

        true_name = classes[true_lbl] if true_lbl < len(classes) else str(true_lbl)
        pred_name = classes[pred_cls] if pred_cls < len(classes) else str(pred_cls)
        ok = '✅' if true_lbl == pred_cls else '❌'

        axes[i, 0].imshow(orig);  axes[i, 0].axis('off')
        axes[i, 0].set_title(f'Original\nTrue: {true_name}', fontsize=9)
        axes[i, 1].imshow(cam, cmap='jet');  axes[i, 1].axis('off')
        axes[i, 1].set_title(f'GradCAM++\nConf: {probs[pred_cls]:.2%}',
                              fontsize=9)
        axes[i, 2].imshow(over);  axes[i, 2].axis('off')
        axes[i, 2].set_title(f'Overlay {ok}\nPred: {pred_name}', fontsize=9)

    plt.tight_layout()
    out = f'models/{modality}_roi_gradcam.png'
    plt.savefig(out, dpi=130, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved → {out}')


for modality in ['xray', 'ct', 'mri']:
    visualize_roi(modality)


🔍 XRAY — GradCAM++ ROI
✅ Saved → models/xray_roi_gradcam.png

🔍 CT — GradCAM++ ROI
✅ Saved → models/ct_roi_gradcam.png

🔍 MRI — GradCAM++ ROI
✅ Saved → models/mri_roi_gradcam.png


In [10]:
# ── CELL 10 — Save Summary & Per-Class Report ────────────────────────────────
summary = {}

for modality in ['xray', 'ct', 'mri']:
    r = results[modality]
    if not r['trained']:
        summary[modality] = {
            'test_accuracy': None,
            'classes': list(r['classes']),
            'note': 'Skipped.'
        }
        continue

    preds = np.argmax(r['test_probs'], axis=1)
    acc   = float((preds == r['test_labels']).mean())
    prec, rec, f1, sup = precision_recall_fscore_support(
        r['test_labels'], preds, zero_division=0)

    summary[modality] = {
        'test_accuracy': acc,
        'classes': list(r['classes']),
        'per_class': {
            cls: {
                'precision': float(prec[i]),
                'recall':    float(rec[i]),
                'f1':        float(f1[i]),
                'support':   int(sup[i]),
            }
            for i, cls in enumerate(r['classes'])
        }
    }

with open(MODEL_DIR / 'summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print('=' * 60)
print('FINAL MODEL SUMMARY')
print('=' * 60)
for mod, info in summary.items():
    if info['test_accuracy'] is not None:
        print(f'  {mod.upper():6s}: Test Acc = {info["test_accuracy"]*100:.2f}%  '
              f'Classes = {info["classes"]}')
    else:
        print(f'  {mod.upper():6s}: {info.get("note", "skipped")}')
print('=' * 60)

print()
print('📊 PER-CLASS PERFORMANCE')
print(f'{"Modality":<8} {"Class":<34} {"Prec":>6} {"Rec":>6} {"F1":>6} {"N":>7}')
print('-' * 68)
for modality in ['xray', 'ct', 'mri']:
    r = results[modality]
    if not r['trained']:
        print(f'{modality.upper():<8} — skipped')
        print()
        continue
    preds = np.argmax(r['test_probs'], axis=1)
    prec, rec, f1, sup = precision_recall_fscore_support(
        r['test_labels'], preds, zero_division=0)
    for i, cls in enumerate(r['classes']):
        print(f'{modality.upper():<8} {cls:<34} '
              f'{prec[i]:>6.3f} {rec[i]:>6.3f} {f1[i]:>6.3f} {int(sup[i]):>7d}')
    print()

print('\n✅ All outputs saved → models/')
for fp in sorted(Path('models').glob('*.pth')):
    print(f'  {fp.name}  ({fp.stat().st_size / 1024**2:.1f} MB)')

FINAL MODEL SUMMARY
  XRAY  : Test Acc = 94.55%  Classes = ['COVID', 'Lung_Opacity', 'Normal', 'Viral Pneumonia']
  CT    : Test Acc = 73.02%  Classes = ['adenocarcinoma', 'large cell carcinoma', 'normal', 'squamous cell carcinoma']
  MRI   : Test Acc = 94.38%  Classes = ['glioma', 'meningioma', 'notumor', 'pituitary']

📊 PER-CLASS PERFORMANCE
Modality Class                                Prec    Rec     F1       N
--------------------------------------------------------------------
XRAY     COVID                               0.980  0.970  0.975     542
XRAY     Lung_Opacity                        0.946  0.888  0.916     901
XRAY     Normal                              0.929  0.969  0.949    1528
XRAY     Viral Pneumonia                     0.985  0.955  0.970     201

CT       adenocarcinoma                      0.891  0.475  0.620     120
CT       large cell carcinoma                0.474  0.882  0.616      51
CT       normal                              0.930  0.981  0.955      54
